# Experimente — Detecția Neconcordanțelor

Notebook pentru rularea și evaluarea prompturilor de detecție a neconcordanțelor (incongruities).

**Task:** Identifică dacă răspunsurile voicebotului conțin neconcordanțe și clasifică tipul acestora.

**Output model:**
```json
{
  "has_incongruity": true|false,
  "incongruity_type": "<tip>|null",
  "confidence": "high|medium|low",
  "reasoning": "<justificare>"
}
```

**Tipuri de neconcordanțe:** `incomplet`, `irelevant`, `contradictoriu`, `nealiniat_context`, `halucinatie`

**Structură celule:**
- [0] Setup
- [1] Config experiment
- [2] Prompt renderer + funcții utilitare
- [3] Init model
- [4] Test prompt
- [5] Rulare completă
- [6] Metrici și rezultate
- [7] Salvare experiment
- [8] Comparație experimente

In [ ]:
# [0] Setup — paths și imports
import os

os.chdir(
    r"C:\Users\Matebook 14s\Documents"
    r"\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-"
)

print("Working directory:", os.getcwd())

Working directory: C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-


In [ ]:
# [0b] Imports și paths
import json
import time
import re
from pathlib import Path
from collections import defaultdict

import numpy as np
from jinja2 import Environment, FileSystemLoader
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, classification_report
from tabulate import tabulate

# ── Paths ──────────────────────────────────────────────────────────────────
PROMPTS_DIR  = Path("prompts/incongruities")
CONFIGS_DIR  = Path("configs")
DATASET_PATH = Path("data/master_dataset_refined_180.json")
OUTPUT_DIR   = Path(r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs_incongruities")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Labels ─────────────────────────────────────────────────────────────────
# Tipurile de neconcordanțe + eticheta specială "none" pentru absența lor
with open(CONFIGS_DIR / "incongruities_definitions.json", encoding="utf-8") as f:
    _defs = json.load(f)
INC_TYPES   = [l["name"] for l in _defs["labels"]]  # tipurile de neconcordanțe
VALID_TYPES = set(INC_TYPES)

# ── Dataset ────────────────────────────────────────────────────────────────
with open(DATASET_PATH, encoding="utf-8") as f:
    DATASET = json.load(f)["conversations"]

# Statistici dataset
n_with_inc = sum(1 for c in DATASET if c["incongruities"])
n_without  = sum(1 for c in DATASET if not c["incongruities"])

print(f"✓ Dataset: {len(DATASET)} conversații")
print(f"  Cu neconcordanțe:    {n_with_inc}")
print(f"  Fără neconcordanțe:  {n_without}")
print(f"✓ Tipuri: {INC_TYPES}")
print(f"✓ Output: {OUTPUT_DIR}")

✓ Dataset: 180 conversații
  Cu neconcordanțe:    36
  Fără neconcordanțe:  144
✓ Tipuri: ['incomplet', 'irelevant', 'contradictoriu', 'nealiniat_context', 'halucinatie']
✓ Output: C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs_incongruities


In [ ]:
# [1] Configurație experiment — editează aici
# ═══════════════════════════════════════════════════════════

MODEL          = "aya_expanse_8b"       # openai_o3 | gemini_2.5_flash | aya_expanse_8b | rollama2_7b | roberta_encoder
LANG           = "en"              # ro | en
PROMPT_VERSION = "v4"              # v1 | v2 | v3 | v4

# Rulează pe tot datasetul sau doar pe primele N conversații (None = tot)
MAX_CONVERSATIONS = None

# ═══════════════════════════════════════════════════════════
EXP_NAME = f"inc_{MODEL}__{LANG}__{PROMPT_VERSION}"
EXP_FILE = OUTPUT_DIR / f"exp_{EXP_NAME}.json"

conversations = DATASET[:MAX_CONVERSATIONS] if MAX_CONVERSATIONS else DATASET

print(f"Experiment : {EXP_NAME}")
print(f"Model      : {MODEL}")
print(f"Limbă      : {LANG}")
print(f"Prompt     : {LANG}_incongruities_{PROMPT_VERSION}.jinja")
print(f"Dataset    : {len(conversations)} conversații")
print(f"Output     : {EXP_FILE}")

Experiment : inc_aya_expanse_8b__en__v4
Model      : aya_expanse_8b
Limbă      : en
Prompt     : en_incongruities_v4.jinja
Dataset    : 180 conversații
Output     : C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs_incongruities\exp_inc_aya_expanse_8b__en__v4.json


In [ ]:
# [2] Prompt renderer + parser + funcții utilitare
# ─────────────────────────────────────────────────────────────────────────

env = Environment(loader=FileSystemLoader(str(PROMPTS_DIR)))

def render_prompt(lang, conversation, version, few_shot_examples=None):
    """
    Renderizează promptul Jinja2 cu conversația și opțional cu few-shot examples.
    
    Args:
        lang: 'ro' sau 'en'
        conversation: lista de turn-uri
        version: 'v1', 'v2', 'v3', 'v4'
        few_shot_examples: lista de exemple pentru few-shot (doar pentru v4)
    """
    template_name = f"{lang}_incongruities_{version}.jinja"
    env = Environment(loader=FileSystemLoader(PROMPTS_DIR))
    template = env.get_template(template_name)
    
    # Renderizează cu sau fără few-shot examples
    if few_shot_examples:
        return template.render(
            conversation=conversation,
            few_shot_examples=few_shot_examples
        )
    else:
        return template.render(conversation=conversation)

# ── Response parser ────────────────────────────────────────────────────────
def parse_response(raw: str):
    """
    Returnează (has_incongruity, incongruity_type, confidence, reasoning, parse_failed).
    Gestionează JSON înfășurat în ```json ... ```.
    """
    text = raw.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    try:
        data = json.loads(text)
        has_inc  = data.get("has_incongruity")
        inc_type = data.get("incongruity_type")
        # Validare
        if not isinstance(has_inc, bool):
            return None, None, None, None, True
        if has_inc and inc_type not in VALID_TYPES:
            return has_inc, inc_type, data.get("confidence"), data.get("reasoning"), True
        if not has_inc:
            inc_type = None
        return (
            has_inc,
            inc_type,
            data.get("confidence"),
            data.get("reasoning"),
            False,
        )
    except json.JSONDecodeError:
        return None, None, None, None, True

# ── Helper: eticheta binară din dataset ───────────────────────────────────
def get_dataset_labels(conv: dict):
    """
    Returnează (has_incongruity, incongruity_type) din dataset.
    Dacă lista incongruities e goală → (False, None)
    Dacă are cel puțin un element → (True, primul_tip)
    """
    inc_list = conv.get("incongruities", [])
    if not inc_list:
        return False, None
    return True, inc_list[0]

# ── Metrici ────────────────────────────────────────────────────────────────
def compute_metrics_binary(results: list):
    """Metrici pentru detecția binară (are/nu are neconcordanță)."""
    valid = [r for r in results if not r["parse_failed"] and r["predicted_has_incongruity"] is not None]
    if not valid:
        return {}
    y_true = [r["dataset_has_incongruity"] for r in valid]
    y_pred = [r["predicted_has_incongruity"] for r in valid]
    return {
        "binary_accuracy": round(accuracy_score(y_true, y_pred), 4),
        "binary_f1":       round(f1_score(y_true, y_pred, average="binary", pos_label=True, zero_division=0), 4),
        "n_valid":         len(valid),
    }

def compute_metrics_type(results: list):
    """Metrici pentru clasificarea tipului de neconcordanță (doar pe conversațiile cu neconcordanță)."""
    # Doar conversațiile unde atât dataset-ul cât și modelul detectează neconcordanță
    valid = [
        r for r in results
        if not r["parse_failed"]
        and r["dataset_has_incongruity"] == True
        and r["predicted_has_incongruity"] == True
        and r["predicted_incongruity_type"] is not None
    ]
    if not valid:
        return {}
    y_true = [r["dataset_incongruity_type"] for r in valid]
    y_pred = [r["predicted_incongruity_type"] for r in valid]
    return {
        "type_accuracy": round(accuracy_score(y_true, y_pred), 4),
        "type_macro_f1": round(f1_score(y_true, y_pred, average="macro", labels=INC_TYPES, zero_division=0), 4),
        "n_type_valid":  len(valid),
    }

# ── Tabel metrici sumar ────────────────────────────────────────────────────
def print_metrics_summary(results: list, exp_name: str):
    fails = sum(1 for r in results if r["parse_failed"])
    total = len(results)
    lats  = [r["latency_ms"] for r in results if r["latency_ms"] > 0]

    mb = compute_metrics_binary(results)
    mt = compute_metrics_type(results)

    rows = [
        ["Experiment",        exp_name],
        ["Binary Accuracy",   f"{mb.get('binary_accuracy', 0)*100:.1f}%"],
        ["Binary F1",         f"{mb.get('binary_f1', 0)*100:.1f}%"],
        ["Type Accuracy",     f"{mt.get('type_accuracy', 0)*100:.1f}%"],
        ["Type Macro F1",     f"{mt.get('type_macro_f1', 0)*100:.1f}%"],
        ["Parse failures",    f"{fails}/{total}  ({fails/total*100:.1f}%)"],
        ["Latență medie",     f"{np.mean(lats):.0f}ms" if lats else "—"],
        ["Latență p95",       f"{np.percentile(lats, 95):.0f}ms" if lats else "—"],
        ["N valid binary",    mb.get('n_valid', 0)],
        ["N valid type",      mt.get('n_type_valid', 0)],
    ]
    print(tabulate(rows, tablefmt="simple", headers=["Metrică", "Valoare"]))
    return {**mb, **mt, "n_failed": fails}

# ── Tabel predicții ────────────────────────────────────────────────────────
def print_predictions_table(results: list):
    rows = []
    for r in results:
        if r["parse_failed"]:
            status = "FAIL"
        elif r["predicted_has_incongruity"] == r["dataset_has_incongruity"]:
            if r["dataset_has_incongruity"]:
                # Are neconcordanță — verifică și tipul
                if r["predicted_incongruity_type"] == r["dataset_incongruity_type"]:
                    status = f"✓  {r['predicted_incongruity_type']}"
                else:
                    status = f"✓bin ✗tip → {r['predicted_incongruity_type']} (real: {r['dataset_incongruity_type']})"
            else:
                status = "✓  none"
        else:
            pred = r['predicted_incongruity_type'] or 'none'
            real = r['dataset_incongruity_type'] or 'none'
            status = f"✗  {pred}  (real: {real})"
        rows.append([
            r["conversation_id"],
            r["dataset_incongruity_type"] or "none",
            status,
            r.get("confidence", "—"),
            f"{r['latency_ms']:.0f}ms",
        ])
    print(tabulate(rows,
                   headers=["conv_id", "dataset_label", "predicție", "confidence", "latență"],
                   tablefmt="rounded_outline"))

# ── Top erori ──────────────────────────────────────────────────────────────
def print_top_errors(results: list):
    errors = defaultdict(lambda: defaultdict(int))
    for r in results:
        if r["parse_failed"]:
            continue
        true_type = r["dataset_incongruity_type"] or "none"
        pred_type = r["predicted_incongruity_type"] or "none"
        if true_type != pred_type:
            errors[true_type][pred_type] += 1

    rows = [
        [true, pred, cnt]
        for true, preds in errors.items()
        for pred, cnt in preds.items()
    ]
    rows.sort(key=lambda r: -r[2])
    if rows:
        print(tabulate(rows[:10],
                       headers=["Tip real", "Prezis ca", "# greșeli"],
                       tablefmt="simple"))
    else:
        print("  0 erori.")

print("✓ Funcții utilitare încărcate.")

✓ Funcții utilitare încărcate.


In [5]:
# [3] Inițializează modelul selectat în [1]
# ─────────────────────────────────────────────────────────────────────────
from dotenv import load_dotenv
load_dotenv()

if MODEL == "openai_o3":
    from openai import OpenAI
    _client = OpenAI()

    def call_model(prompt: str) -> str:
        r = _client.chat.completions.create(
            model="o3",
            messages=[{"role": "user", "content": prompt}],
        )
        return r.choices[0].message.content

elif MODEL == "gemini_2.5_flash":
    import os
    from google.genai import Client
    _client = Client(api_key=os.environ["GEMINI_API_KEY"])

    def call_model(prompt: str) -> str:
        response = _client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt,
        )
        return response.text

elif MODEL == "aya_expanse_8b":
    import ollama as _ollama

    def call_model(prompt: str) -> str:
        r = _ollama.chat(
            model="aya-expanse:8b",
            messages=[{"role": "user", "content": prompt}],
        )
        return r["message"]["content"]

elif MODEL == "rollama2_7b":
    import ollama as _ollama

    def call_model(prompt: str) -> str:
        r = _ollama.chat(
            model="rollama2:7b-instruct",
            messages=[{"role": "user", "content": prompt}],
        )
        return r["message"]["content"]

elif MODEL == "roberta_encoder":
    # XLM-RoBERTa XNLI — zero-shot classification NLI
    # Notă: pentru incongruities, RoBERTa face NLI între conversație și tipul de neconcordanță
    from transformers import pipeline as hf_pipeline
    _pipe = hf_pipeline(
        "zero-shot-classification",
        model="joeddav/xlm-roberta-large-xnli",
        device=-1,
    )

    def _extract_conversation_text(prompt: str) -> str:
        lines = prompt.split("\n")
        conv_lines, in_conv = [], False
        for line in lines:
            stripped = line.strip()
            if "Convers" in stripped and stripped.startswith("#"):
                in_conv = True
                continue
            if in_conv and stripped.startswith("#"):
                break
            if in_conv and stripped:
                conv_lines.append(stripped)
        return " ".join(conv_lines)

    def call_model(prompt: str) -> str:
        text = _extract_conversation_text(prompt)
        candidate_labels = INC_TYPES + ["fara_neconcordanta"]
        result = _pipe(text, candidate_labels=candidate_labels)
        predicted = result["labels"][0]
        score     = result["scores"][0]
        conf = "high" if score > 0.7 else "medium" if score > 0.4 else "low"
        has_inc  = predicted != "fara_neconcordanta"
        inc_type = predicted if has_inc else None
        return json.dumps({
            "has_incongruity":   has_inc,
            "incongruity_type":  inc_type,
            "confidence":        conf,
            "reasoning":         f"zero-shot score: {score:.4f}",
        })

else:
    raise ValueError(f"Model necunoscut: {MODEL}")

print(f"✓ Model inițializat: {MODEL}")

✓ Model inițializat: aya_expanse_8b


In [ ]:
# Debug — vezi raw output de la Aya
few_shot_examples = []
if PROMPT_VERSION == "v4":
    few_shot_file = "few_shot_examples_incongruities.json" if LANG == "ro" else "few_shot_examples_incongruities.json"
    try:
        with open(CONFIGS_DIR / few_shot_file, encoding="utf-8") as f:
            few_shot_examples = json.load(f).get("examples", [])
        print(f"✓ {len(few_shot_examples)} exemple few-shot încărcate")
    except FileNotFoundError:
        print(f"⚠ {few_shot_file} nu a fost găsit")

sample_conv = conversations[0]
prompt = render_prompt(LANG, sample_conv["turns"], PROMPT_VERSION, few_shot_examples)

print(f"Lungime prompt: {len(prompt)} caractere")
print("─" * 60)

raw = call_model(prompt)
print("RAW OUTPUT:")
print(repr(raw))
print("─" * 60)
print(raw)

⚠ few_shot_examples_incongruities_en.json nu a fost găsit
Lungime prompt: 9671 caractere
────────────────────────────────────────────────────────────


In [6]:
# [4] Test prompt pe prima conversație
sample_conv = conversations[0]

# Încarcă few-shot examples pentru v4
few_shot_examples = []
if PROMPT_VERSION == "v4":
    few_shot_path = CONFIGS_DIR / "few_shot_examples_incongruities.json"
    if few_shot_path.exists():
        with open(few_shot_path, encoding='utf-8') as f:
            few_shot_data = json.load(f)
            few_shot_examples = few_shot_data.get('examples', [])

# Render prompt cu few-shot examples
sample_prompt = render_prompt(LANG, sample_conv["turns"], PROMPT_VERSION, few_shot_examples)

ds_has_inc, ds_type = get_dataset_labels(sample_conv)

print(f"── Conversație: {sample_conv['conversation_id']}")
print(f"── dataset has_incongruity: {ds_has_inc}")
print(f"── dataset incongruity_type: {ds_type}")
print("─" * 60)
print(sample_prompt)
print("─" * 60)

raw = call_model(sample_prompt)
print("RAW:", repr(raw))
print("PARSED:", parse_response(raw))

── Conversație: conv_simple_0001
── dataset has_incongruity: False
── dataset incongruity_type: None
────────────────────────────────────────────────────────────
You are a quality auditor for a Romanian-language banking voicebot system.

Your task is to analyze the conversation below and determine whether the voicebot's responses contain an incongruity according to the definitions in this prompt.

IMPORTANT:
Do not evaluate the conversation based on real banking rules, real security procedures, or assumptions about what a voicebot could do in reality.
Evaluate only the text of the conversation and the labeling rules below.

## Objective

For each conversation:
1. determine whether there is an incongruity in the ASSISTANT messages;
2. if there is one, choose a single dominant type from:
`incomplet`, `irelevant`, `contradictoriu`, `nealiniat_context`, `halucinatie`.

## Examples

Here are examples of correct classifications to better understand each type of incongruity:


### Example 1



In [ ]:
# [5] Rulare completă
results = []
total   = len(conversations)

for i, conv in enumerate(conversations):
    conv_id                    = conv["conversation_id"]
    ds_has_inc, ds_type        = get_dataset_labels(conv)
    turns                      = conv["turns"]

    prompt = render_prompt(LANG, turns, PROMPT_VERSION)

    start = time.monotonic()
    try:
        raw        = call_model(prompt)
        latency_ms = (time.monotonic() - start) * 1000
        has_inc, inc_type, confidence, reasoning, parse_failed = parse_response(raw)
    except Exception as e:
        latency_ms   = (time.monotonic() - start) * 1000
        raw          = str(e)
        has_inc      = None
        inc_type     = None
        confidence   = None
        reasoning    = None
        parse_failed = True

    result = {
        "conversation_id":            conv_id,
        "dataset_has_incongruity":     ds_has_inc,
        "dataset_incongruity_type":    ds_type,
        "model_name":                  MODEL,
        "prompt_lang":                 LANG,
        "prompt_version":              PROMPT_VERSION,
        "predicted_has_incongruity":   has_inc,
        "predicted_incongruity_type":  inc_type,
        "confidence":                  confidence,
        "reasoning":                   reasoning,
        "latency_ms":                  round(latency_ms, 1),
        "parse_failed":                parse_failed,
        "raw_response":                raw,
    }
    results.append(result)

    # Progress live
    if parse_failed:
        status = "FAIL"
    elif has_inc == ds_has_inc:
        if ds_has_inc:
            status = f"✓bin {'✓tip' if inc_type == ds_type else f'✗tip→{inc_type}'}"
        else:
            status = "✓  none"
    else:
        status = f"✗  pred={inc_type or 'none'}  real={ds_type or 'none'}"

    print(f"[{i+1:>3}/{total}]  {conv_id:<25} {status:<50} {latency_ms:.0f}ms")

print(f"\n✓ Terminat. {len(results)} predicții.")

[  1/180]  conv_simple_0001          FAIL                                               259407ms
[  2/180]  conv_simple_0002          FAIL                                               70297ms
[  3/180]  conv_simple_0003          FAIL                                               68109ms


In [ ]:
# [6] Metrici și rezultate
print(f"{'═'*60}")
print(f"  METRICI SUMAR — {EXP_NAME}")
print(f"{'═'*60}\n")
metrics = print_metrics_summary(results, EXP_NAME)

print(f"\n{'─'*60}")
print(f"  PREDICȚII")
print(f"{'─'*60}\n")
print_predictions_table(results)

print(f"\n{'─'*60}")
print(f"  TOP GREȘELI")
print(f"{'─'*60}\n")
print_top_errors(results)

════════════════════════════════════════════════════════════
  METRICI SUMAR — inc_gemini_2.5_flash__en__v4
════════════════════════════════════════════════════════════

Metrică          Valoare
---------------  ----------------------------
Experiment       inc_gemini_2.5_flash__en__v4
Binary Accuracy  91.7%
Binary F1        80.0%
Type Accuracy    76.7%
Type Macro F1    75.9%
Parse failures   0/180  (0.0%)
Latență medie    5968ms
Latență p95      12869ms
N valid binary   180
N valid type     30

────────────────────────────────────────────────────────────
  PREDICȚII
────────────────────────────────────────────────────────────

╭───────────────────┬───────────────────┬─────────────────────────────────────────────────┬──────────────┬───────────╮
│ conv_id           │ dataset_label     │ predicție                                       │ confidence   │ latență   │
├───────────────────┼───────────────────┼─────────────────────────────────────────────────┼──────────────┼───────────┤
│ conv_

In [ ]:
# [7] Salvare experiment
payload = {
    "experiment": {
        "name":           EXP_NAME,
        "task":           "incongruities",
        "model":          MODEL,
        "lang":           LANG,
        "prompt_version": PROMPT_VERSION,
        "prompt_file":    f"{LANG}_incongruities_{PROMPT_VERSION}.jinja",
        "n_conversations":len(results),
        "timestamp":      time.strftime("%Y%m%d_%H%M%S"),
    },
    "metrics":    metrics,
    "predictions": results,
}

with open(EXP_FILE, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print(f"✓ Salvat în: {EXP_FILE}")
print(f"  Binary Accuracy: {metrics.get('binary_accuracy', 0)*100:.1f}%")
print(f"  Binary F1:       {metrics.get('binary_f1', 0)*100:.1f}%")
print(f"  Type Accuracy:   {metrics.get('type_accuracy', 0)*100:.1f}%")
print(f"  Parse fails:     {metrics.get('n_failed', 0)}/{len(results)}")

✓ Salvat în: C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs_incongruities\exp_inc_gemini_2.5_flash__en__v4.json
  Binary Accuracy: 91.7%
  Binary F1:       80.0%
  Type Accuracy:   76.7%
  Parse fails:     0/180


### Analiza Experimente

In [10]:
# [8] Comparație experimente
# ── Setează experimentele de comparat ──────────────────────────────────────
EXP_A = "inc_gemini_2.5_flash__en__v4"
EXP_B = "inc_gemini_2.5_flash__en__v3"
# ──────────────────────────────────────────────────────────────────────────

def load_exp(name: str) -> dict:
    path = OUTPUT_DIR / f"exp_{name}.json"
    with open(path, encoding="utf-8") as f:
        return json.load(f)

exp_a = load_exp(EXP_A)
exp_b = load_exp(EXP_B)

ma = exp_a["metrics"]
mb = exp_b["metrics"]

def d(a, b):
    if a is None or b is None: return "—"
    diff = a - b
    return f"{'+'if diff>=0 else ''}{diff*100:.1f}%"

print(f"{'═'*70}")
print(f"  COMPARAȚIE: {EXP_A}  vs  {EXP_B}")
print(f"{'═'*70}\n")

cmp_rows = [
    ["Binary Accuracy", f"{ma.get('binary_accuracy',0)*100:.1f}%", f"{mb.get('binary_accuracy',0)*100:.1f}%", d(ma.get('binary_accuracy'), mb.get('binary_accuracy'))],
    ["Binary F1",       f"{ma.get('binary_f1',0)*100:.1f}%",       f"{mb.get('binary_f1',0)*100:.1f}%",       d(ma.get('binary_f1'),       mb.get('binary_f1'))],
    ["Type Accuracy",   f"{ma.get('type_accuracy',0)*100:.1f}%",   f"{mb.get('type_accuracy',0)*100:.1f}%",   d(ma.get('type_accuracy'),   mb.get('type_accuracy'))],
    ["Type Macro F1",   f"{ma.get('type_macro_f1',0)*100:.1f}%",   f"{mb.get('type_macro_f1',0)*100:.1f}%",   d(ma.get('type_macro_f1'),   mb.get('type_macro_f1'))],
]
print(tabulate(cmp_rows, headers=["Metrică", EXP_A, EXP_B, "Δ (A−B)"], tablefmt="rounded_outline"))

# ── Predicții unde diferă ──────────────────────────────────────────────────
preds_a = {r["conversation_id"]: r for r in exp_a["predictions"]}
preds_b = {r["conversation_id"]: r for r in exp_b["predictions"]}

diff_rows = []
for cid, ra in preds_a.items():
    rb = preds_b.get(cid)
    if rb is None:
        continue
    if ra["predicted_has_incongruity"] != rb["predicted_has_incongruity"] or \
       ra["predicted_incongruity_type"] != rb["predicted_incongruity_type"]:
        ds_type = ra["dataset_incongruity_type"] or "none"
        ca = "✓" if ra["predicted_incongruity_type"] == ra["dataset_incongruity_type"] else "✗"
        cb = "✓" if rb["predicted_incongruity_type"] == rb["dataset_incongruity_type"] else "✗"
        diff_rows.append([
            cid,
            ds_type,
            f"{ca} {ra['predicted_incongruity_type'] or 'none'}",
            f"{cb} {rb['predicted_incongruity_type'] or 'none'}",
        ])

if diff_rows:
    print(f"\n  Predicții diferite: {len(diff_rows)}")
    print(tabulate(diff_rows,
                   headers=["conv_id", "dataset", EXP_A, EXP_B],
                   tablefmt="simple"))
else:
    print("\n  Predicții identice între cele două experimente.")

══════════════════════════════════════════════════════════════════════
  COMPARAȚIE: inc_gemini_2.5_flash__en__v4  vs  inc_gemini_2.5_flash__en__v3
══════════════════════════════════════════════════════════════════════

╭─────────────────┬────────────────────────────────┬────────────────────────────────┬───────────╮
│ Metrică         │ inc_gemini_2.5_flash__en__v4   │ inc_gemini_2.5_flash__en__v3   │ Δ (A−B)   │
├─────────────────┼────────────────────────────────┼────────────────────────────────┼───────────┤
│ Binary Accuracy │ 91.7%                          │ 91.6%                          │ +0.0%     │
│ Binary F1       │ 80.0%                          │ 78.9%                          │ +1.1%     │
│ Type Accuracy   │ 76.7%                          │ 78.6%                          │ -1.9%     │
│ Type Macro F1   │ 75.9%                          │ 78.7%                          │ -2.8%     │
╰─────────────────┴────────────────────────────────┴────────────────────────────────┴─────────

In [5]:
# ════════════════════════════════════════════════════════════
# ANALIZĂ ERORI — Incongruities
# Rulează după ce ai cel puțin un experiment salvat
# ════════════════════════════════════════════════════════════
import json
from pathlib import Path
from collections import defaultdict
from tabulate import tabulate

# !! Schimbă cu experimentul pe care vrei să îl analizezi
EXP_TO_ANALYZE = "inc_openai_o3__ro__v1"

OUTPUT_DIR = Path(r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs_incongruities")

with open(OUTPUT_DIR / f"exp_{EXP_TO_ANALYZE}.json", encoding="utf-8") as f:
    exp = json.load(f)

predictions = exp["predictions"]
total  = len(predictions)
fails  = sum(1 for r in predictions if r["parse_failed"])

print(f"Experiment: {EXP_TO_ANALYZE}")
print(f"Total: {total}  |  Parse fails: {fails}  |  Valid: {total - fails}")

Experiment: inc_openai_o3__ro__v1
Total: 180  |  Parse fails: 0  |  Valid: 180


In [13]:
# ════════════════════════════════════════════════════════════
# SUMAR METRICI
# ════════════════════════════════════════════════════════════
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

valid_binary = [r for r in predictions if not r["parse_failed"] and r["predicted_has_incongruity"] is not None]
y_true_bin   = [r["dataset_has_incongruity"]   for r in valid_binary]
y_pred_bin   = [r["predicted_has_incongruity"]  for r in valid_binary]

valid_type   = [r for r in predictions
                if not r["parse_failed"]
                and r["dataset_has_incongruity"] == True
                and r["predicted_has_incongruity"] == True
                and r["predicted_incongruity_type"] is not None]
y_true_type  = [r["dataset_incongruity_type"]   for r in valid_type]
y_pred_type  = [r["predicted_incongruity_type"]  for r in valid_type]

INC_TYPES = ["incomplet", "irelevant", "contradictoriu", "nealiniat_context", "halucinatie"]

print(f"{'═'*60}")
print(f"  METRICI — {EXP_TO_ANALYZE}")
print(f"{'═'*60}")

rows = [
    ["Binary Accuracy",  f"{accuracy_score(y_true_bin, y_pred_bin)*100:.1f}%" if y_true_bin else "—"],
    ["Binary F1",        f"{f1_score(y_true_bin, y_pred_bin, average='binary', pos_label=True, zero_division=0)*100:.1f}%" if y_true_bin else "—"],
    ["Type Accuracy",    f"{accuracy_score(y_true_type, y_pred_type)*100:.1f}%" if y_true_type else "—"],
    ["Type Macro F1",    f"{f1_score(y_true_type, y_pred_type, average='macro', labels=INC_TYPES, zero_division=0)*100:.1f}%" if y_true_type else "—"],
    ["N valid binary",   len(valid_binary)],
    ["N valid type",     len(valid_type)],
    ["Parse fails",      fails],
]
print(tabulate(rows, headers=["Metrică", "Valoare"], tablefmt="simple"))

════════════════════════════════════════════════════════════


NameError: name 'EXP_TO_ANALYZE' is not defined

In [14]:
# ════════════════════════════════════════════════════════════
# ANALIZĂ GREȘELI — toate experimentele incongruities
# Afișează greșelile per experiment cu reasoning complet
# ════════════════════════════════════════════════════════════
import json
from pathlib import Path
from tabulate import tabulate

OUTPUT_DIR = Path(r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs_incongruities")

with open(DATASET_PATH, encoding="utf-8") as f:
    data = json.load(f)
conv_dict = {c["conversation_id"]: c for c in data["conversations"]}

for exp_file in sorted(OUTPUT_DIR.glob("exp_inc_*.json")):
    with open(exp_file, encoding="utf-8") as f:
        exp = json.load(f)

    exp_name    = exp["experiment"]["name"]
    predictions = exp["predictions"]
    errors      = []

    for r in predictions:
        if r["parse_failed"]:
            continue

        ds_has  = r["dataset_has_incongruity"]
        pr_has  = r["predicted_has_incongruity"]
        ds_type = r["dataset_incongruity_type"]
        pr_type = r["predicted_incongruity_type"]

        if ds_has != pr_has or (ds_has and ds_type != pr_type):
            error_type = "FP" if (pr_has and not ds_has) else "FN" if (not pr_has and ds_has) else "TIP_GRESIT"
            errors.append({
                "conv_id":        r["conversation_id"],
                "error_type":     error_type,
                "dataset_type":   ds_type or "none",
                "predicted_type": pr_type or "none",
                "reasoning":      r.get("reasoning", ""),
            })

    print(f"\n{'═'*80}")
    print(f"  {exp_name} — {len(errors)} greșeli")
    print(f"{'═'*80}")

    if not errors:
        print("  0 greșeli.")
        continue

    rows = [
        [
            e["conv_id"],
            e["error_type"],
            e["dataset_type"],
            e["predicted_type"],
            e["reasoning"],
        ]
        for e in errors
    ]
    print(tabulate(rows,
                   headers=["conv_id", "Tip eroare", "Real", "Prezis", "Reasoning"],
                   tablefmt="simple",
                   maxcolwidths=[25, 12, 20, 20, 80]))


════════════════════════════════════════════════════════════════════════════════
  inc_aya_expanse_8b__en__v1 — 55 greșeli
════════════════════════════════════════════════════════════════════════════════
conv_id            Tip eroare    Real               Prezis             Reasoning
-----------------  ------------  -----------------  -----------------  --------------------------------------------------------------------------------
conv_simple_0041   FP            none               nealiniat_context  The bot ignores the user's explicit request to speak with an agent and instead
                                                                       redirects to a different topic (soliciting an extras de cont).
conv_simple_0059   FP            none               irelevant          The assistant's response does not address the user's question about the weather
                                                                       and instead offers services unrelated to the initial req

In [15]:
# ════════════════════════════════════════════════════════════
# CLASAMENT GREȘELI — cele mai greșite conversații
# ════════════════════════════════════════════════════════════
from collections import defaultdict
from tabulate import tabulate

# Numără câte modele au greșit per conversație
error_count = {cid: len(errors) for cid, errors in all_errors.items()}

# Sortează descrescător
sorted_errors = sorted(error_count.items(), key=lambda x: -x[1])

print(f"{'═'*70}")
print(f"  CLASAMENT — conversații greșite de cele mai multe modele ({VERSION})")
print(f"{'═'*70}\n")

rows = []
for cid, count in sorted_errors:
    conv     = conv_dict.get(cid, {})
    inc_list = conv.get("incongruities", [])
    ds_type  = inc_list[0] if inc_list else "none"
    ds_has   = bool(inc_list)

    # Tipurile de erori la această conversație
    error_types = defaultdict(int)
    for e in all_errors[cid].values():
        error_types[e["error_type"]] += 1

    fp  = error_types.get("FP", 0)
    fn  = error_types.get("FN", 0)
    tip = error_types.get("TIP_GRESIT", 0)

    rows.append([
        cid,
        f"{count}/{n_models}",
        "DA → " + ds_type if ds_has else "NU",
        fp, fn, tip,
    ])

print(tabulate(rows,
               headers=["conv_id", "Greșeli", "Dataset", "FP", "FN", "Tip greșit"],
               tablefmt="rounded_outline"))

# Sumar global
print(f"\n  Sumar:")
print(f"  Greșite de toate {n_models} modelele:   {sum(1 for _, c in sorted_errors if c == n_models)}")
print(f"  Greșite de {n_models-1} modele:          {sum(1 for _, c in sorted_errors if c == n_models-1)}")
print(f"  Greșite de {n_models-2} modele:          {sum(1 for _, c in sorted_errors if c == n_models-2)}")
print(f"  Greșite de 1 model:             {sum(1 for _, c in sorted_errors if c == 1)}")

NameError: name 'all_errors' is not defined

In [16]:
# ════════════════════════════════════════════════════════════
# EXPORT GREȘELI — toate experimentele incongruities
# Salvează CSV-uri cu greșelile per experiment
# ════════════════════════════════════════════════════════════
import json
import csv
from pathlib import Path

OUTPUT_DIR  = Path(r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs_incongruities")
ERRORS_DIR  = Path(r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\errors_incongruities")
ERRORS_DIR.mkdir(parents=True, exist_ok=True)

# Încarcă dataset pentru textul conversațiilor
with open(DATASET_PATH, encoding="utf-8") as f:
    data = json.load(f)
conv_dict = {c["conversation_id"]: c for c in data["conversations"]}

for exp_file in sorted(OUTPUT_DIR.glob("exp_inc_*.json")):
    with open(exp_file, encoding="utf-8") as f:
        exp = json.load(f)

    exp_name    = exp["experiment"]["name"]
    predictions = exp["predictions"]
    errors      = []

    for r in predictions:
        if r["parse_failed"]:
            continue

        ds_has  = r["dataset_has_incongruity"]
        pr_has  = r["predicted_has_incongruity"]
        ds_type = r["dataset_incongruity_type"]
        pr_type = r["predicted_incongruity_type"]

        # Orice greșeală — binară sau de tip
        if ds_has != pr_has or (ds_has and ds_type != pr_type):
            conv = conv_dict.get(r["conversation_id"], {})
            turns_text = " | ".join(
                f"{t['role'].upper()}: {t['text']}"
                for t in conv.get("turns", [])
            )

            error_type = ""
            if ds_has != pr_has:
                error_type = "FP" if pr_has else "FN"
            else:
                error_type = "TIP_GRESIT"

            errors.append({
                "conv_id":          r["conversation_id"],
                "error_type":       error_type,
                "dataset_has":      ds_has,
                "predicted_has":    pr_has,
                "dataset_type":     ds_type or "none",
                "predicted_type":   pr_type or "none",
                "reasoning":        r.get("reasoning", ""),
                "conversatie":      turns_text,
            })

    # Salvează CSV
    if errors:
        csv_path = ERRORS_DIR / f"errors_{exp_name}.csv"
        with open(csv_path, "w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=errors[0].keys())
            writer.writeheader()
            writer.writerows(errors)
        print(f"✓ {exp_name:<40} {len(errors)} greșeli → {csv_path.name}")
    else:
        print(f"  {exp_name:<40} 0 greșeli")

print(f"\n✓ Toate fișierele salvate în: {ERRORS_DIR}")

✓ inc_aya_expanse_8b__en__v1               55 greșeli → errors_inc_aya_expanse_8b__en__v1.csv
✓ inc_aya_expanse_8b__en__v2               41 greșeli → errors_inc_aya_expanse_8b__en__v2.csv
✓ inc_aya_expanse_8b__en__v3               50 greșeli → errors_inc_aya_expanse_8b__en__v3.csv
✓ inc_aya_expanse_8b__ro__v1               57 greșeli → errors_inc_aya_expanse_8b__ro__v1.csv
✓ inc_aya_expanse_8b__ro__v2               60 greșeli → errors_inc_aya_expanse_8b__ro__v2.csv
✓ inc_aya_expanse_8b__ro__v3               109 greșeli → errors_inc_aya_expanse_8b__ro__v3.csv
✓ inc_gemini_2.5_flash__en__v1             58 greșeli → errors_inc_gemini_2.5_flash__en__v1.csv
✓ inc_gemini_2.5_flash__en__v2             49 greșeli → errors_inc_gemini_2.5_flash__en__v2.csv
✓ inc_gemini_2.5_flash__en__v3             21 greșeli → errors_inc_gemini_2.5_flash__en__v3.csv
✓ inc_gemini_2.5_flash__ro__v1             60 greșeli → errors_inc_gemini_2.5_flash__ro__v1.csv
✓ inc_gemini_2.5_flash__ro__v2             44 greșe

In [24]:
# ════════════════════════════════════════════════════════════
# ANALIZĂ COMPARATIVĂ — Toate experimentele incongruities
# Identifică best performing prompt pentru few-shot v4
# ════════════════════════════════════════════════════════════
import json
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

OUTPUT_DIR = Path(r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs_incongruities")

# Găsește toate fișierele de experimente
exp_files = list(OUTPUT_DIR.glob("exp_inc_*.json"))
print(f"Găsite {len(exp_files)} experimente\n")

results = []

for exp_file in sorted(exp_files):
    with open(exp_file, encoding='utf-8') as f:
        exp_data = json.load(f)
    
    # Extrage info experiment
    exp_name = exp_data["experiment"]["name"]
    parts = exp_name.replace("inc_", "").split("__")
    
    if len(parts) != 3:
        continue
    
    model, lang, version = parts
    
    # Extrage predicții
    y_true = []
    y_pred = []
    
    for pred in exp_data["predictions"]:
        if pred.get("parse_failed", False):
            continue
        
        # Ground truth - handle None
        if pred["dataset_has_incongruity"]:
            gt_type = pred.get("dataset_incongruity_type")
            y_true.append(gt_type if gt_type else "none")
        else:
            y_true.append("none")
        
        # Prediction - handle None
        if pred["predicted_has_incongruity"]:
            pred_type = pred.get("predicted_incongruity_type")
            y_pred.append(pred_type if pred_type else "none")
        else:
            y_pred.append("none")
    
    if len(y_true) == 0:
        continue
    
    # Calculează metrici
    accuracy = accuracy_score(y_true, y_pred)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )
    
    results.append({
        'model': model,
        'lang': lang,
        'version': version,
        'accuracy': accuracy,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro,
        'precision_weighted': precision_weighted,
        'recall_weighted': recall_weighted,
        'f1_weighted': f1_weighted,
        'num_samples': len(y_true)
    })

# DataFrame cu rezultate
df = pd.DataFrame(results)
df_sorted = df.sort_values('f1_macro', ascending=False)

print("=" * 120)
print("TOATE EXPERIMENTELE — Sortate după F1-Macro")
print("=" * 120)
print(df_sorted.to_string(index=False))
print("\n")

# Best overall
best = df_sorted.iloc[0]
print("=" * 120)
print("🏆 BEST PERFORMING CONFIGURATION")
print("=" * 120)
print(f"Model:             {best['model']}")
print(f"Language:          {best['lang']}")
print(f"Version:           {best['version']}")
print(f"Accuracy:          {best['accuracy']:.4f}")
print(f"F1-Macro:          {best['f1_macro']:.4f}")
print(f"Precision-Macro:   {best['precision_macro']:.4f}")
print(f"Recall-Macro:      {best['recall_macro']:.4f}")
print("\n")

# Metrici per versiune (medie pe toate modelele și limbile)
print("=" * 120)
print("METRICI MEDII PER VERSIUNE (cross-model, cross-language)")
print("=" * 120)
version_avg = df.groupby('version').agg({
    'accuracy': ['mean', 'std'],
    'f1_macro': ['mean', 'std'],
    'precision_macro': ['mean', 'std'],
    'recall_macro': ['mean', 'std']
}).round(4)
version_avg.columns = ['_'.join(col) for col in version_avg.columns]
version_avg = version_avg.sort_values('f1_macro_mean', ascending=False)
print(version_avg)
print("\n")

best_version = version_avg.index[0]
print(f"✓ Best version overall: {best_version} (F1-Macro mean: {version_avg.loc[best_version, 'f1_macro_mean']:.4f})")
print("\n")

# Metrici per model (medie pe limbă și versiune)
print("=" * 120)
print("METRICI MEDII PER MODEL")
print("=" * 120)
model_avg = df.groupby('model').agg({
    'f1_macro': ['mean', 'std']
}).round(4).sort_values(('f1_macro', 'mean'), ascending=False)
print(model_avg)
print("\n")

# Metrici per limbă
print("=" * 120)
print("METRICI MEDII PER LIMBĂ")
print("=" * 120)
lang_avg = df.groupby('lang').agg({
    'f1_macro': ['mean', 'std']
}).round(4)
print(lang_avg)
print("\n")

# Recomandare pentru v4
print("=" * 120)
print("📋 RECOMANDARE PENTRU FEW-SHOT v4")
print("=" * 120)
print(f"Template de bază:  {best['lang']}_incongruities_{best['version']}.jinja")
print(f"F1-Macro actual:   {best['f1_macro']:.4f}")
print(f"\nPași următori:")
print(f"1. Creează few_shot_examples_incongruities.json cu 6-8 exemple")
print(f"2. Creează {best['lang']}_incongruities_v4.jinja bazat pe {best['version']}")
print(f"3. Adaugă secțiunea cu exemple în prompt")
print(f"4. Rulează experiment: {best['model']}__{best['lang']}__v4")
print("=" * 120)

Găsite 30 experimente

TOATE EXPERIMENTELE — Sortate după F1-Macro
           model lang version  accuracy  precision_macro  recall_macro  f1_macro  precision_weighted  recall_weighted  f1_weighted  num_samples
       openai_o3   en      v3  0.888889         0.760839      0.633466  0.680774            0.881563         0.888889     0.879984          180
gemini_2.5_flash   en      v3  0.882682         0.689881      0.680135  0.678813            0.883440         0.882682     0.881547          179
gemini_2.5_flash   ro      v3  0.888889         0.689771      0.674405  0.676519            0.890229         0.888889     0.888314          180
       openai_o3   en      v2  0.871508         0.670772      0.709203  0.668157            0.878101         0.871508     0.868437          179
       openai_o3   ro      v1  0.844444         0.720538      0.666501  0.617234            0.877087         0.844444     0.836699          180
       openai_o3   ro      v3  0.877778         0.730184      0.56636

In [25]:
# ════════════════════════════════════════════════════════════
# ANALIZA GREȘELILOR — Best performing experiment
# Identifică pattern-urile de greșeli pentru few-shot examples
# ════════════════════════════════════════════════════════════
import json
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict

# Best performing experiment
BEST_EXP = "exp_inc_openai_o3__en__v3.json"
OUTPUT_DIR = Path(r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs_incongruities")
DATASET_PATH = Path(r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\data\master_dataset_refined_180.json")

# Încarcă dataset
with open(DATASET_PATH, encoding='utf-8') as f:
    data = json.load(f)
conv_dict = {c["conversation_id"]: c for c in data["conversations"]}

# Încarcă experiment
with open(OUTPUT_DIR / BEST_EXP, encoding='utf-8') as f:
    exp_data = json.load(f)

# Clasifică greșelile
false_positives = defaultdict(list)  # tip prezis -> conversații
false_negatives = defaultdict(list)  # tip real -> conversații
type_errors = defaultdict(lambda: defaultdict(list))  # tip_real -> tip_prezis -> conversații

for pred in exp_data["predictions"]:
    if pred.get("parse_failed", False):
        continue
    
    conv_id = pred["conversation_id"]
    ds_has = pred["dataset_has_incongruity"]
    pr_has = pred["predicted_has_incongruity"]
    ds_type = pred.get("dataset_incongruity_type") or "none"
    pr_type = pred.get("predicted_incongruity_type") or "none"
    
    conv = conv_dict.get(conv_id, {})
    
    if not ds_has and pr_has:
        # False Positive
        false_positives[pr_type].append({
            "conv_id": conv_id,
            "predicted_type": pr_type,
            "reasoning": pred.get("reasoning", ""),
            "conversation": conv
        })
    elif ds_has and not pr_has:
        # False Negative
        false_negatives[ds_type].append({
            "conv_id": conv_id,
            "real_type": ds_type,
            "reasoning": pred.get("reasoning", ""),
            "conversation": conv
        })
    elif ds_has and pr_has and ds_type != pr_type:
        # Tip greșit
        type_errors[ds_type][pr_type].append({
            "conv_id": conv_id,
            "real_type": ds_type,
            "predicted_type": pr_type,
            "reasoning": pred.get("reasoning", ""),
            "conversation": conv
        })

# ══════════════════════════════════════════════════════════════
# SUMAR GREȘELI
# ══════════════════════════════════════════════════════════════
print("=" * 100)
print("SUMAR GREȘELI — openai_o3__en__v3")
print("=" * 100)

print(f"\n📊 FALSE POSITIVES (n={sum(len(v) for v in false_positives.values())})")
print("-" * 100)
for tip, errors in sorted(false_positives.items(), key=lambda x: len(x[1]), reverse=True):
    print(f"  {tip:20s} → {len(errors):3d} FP")

print(f"\n📊 FALSE NEGATIVES (n={sum(len(v) for v in false_negatives.values())})")
print("-" * 100)
for tip, errors in sorted(false_negatives.items(), key=lambda x: len(x[1]), reverse=True):
    print(f"  {tip:20s} → {len(errors):3d} FN")

print(f"\n📊 TIP GREȘIT (n={sum(len(errors) for tipuri in type_errors.values() for errors in tipuri.values())})")
print("-" * 100)
for real_type, predictions in sorted(type_errors.items()):
    for pred_type, errors in sorted(predictions.items(), key=lambda x: len(x[1]), reverse=True):
        print(f"  {real_type:20s} → {pred_type:20s} ({len(errors):2d} erori)")

# ══════════════════════════════════════════════════════════════
# TOP GREȘELI CU CONVERSAȚII
# ══════════════════════════════════════════════════════════════
print("\n\n" + "=" * 100)
print("TOP 3 FALSE POSITIVES PER TIP (cu conversații)")
print("=" * 100)

for tip, errors in sorted(false_positives.items(), key=lambda x: len(x[1]), reverse=True):
    print(f"\n{'─'*100}")
    print(f"FP: {tip} (total: {len(errors)})")
    print(f"{'─'*100}")
    
    for i, error in enumerate(errors[:3], 1):
        conv = error["conversation"]
        print(f"\n[{i}] {error['conv_id']}")
        print(f"Reasoning: {error['reasoning']}")
        print("Conversație:")
        for turn in conv.get("turns", []):
            print(f"  {turn['role'].upper()}: {turn['text']}")
        print()

print("\n\n" + "=" * 100)
print("TOP 3 FALSE NEGATIVES PER TIP (cu conversații)")
print("=" * 100)

for tip, errors in sorted(false_negatives.items(), key=lambda x: len(x[1]), reverse=True):
    print(f"\n{'─'*100}")
    print(f"FN: {tip} (total: {len(errors)})")
    print(f"{'─'*100}")
    
    for i, error in enumerate(errors[:3], 1):
        conv = error["conversation"]
        print(f"\n[{i}] {error['conv_id']}")
        print(f"Reasoning: {error['reasoning']}")
        print("Conversație:")
        for turn in conv.get("turns", []):
            print(f"  {turn['role'].upper()}: {turn['text']}")
        print()

print("\n\n" + "=" * 100)
print("TOP CONFUZII (TIP GREȘIT)")
print("=" * 100)

all_type_errors = []
for real_type, predictions in type_errors.items():
    for pred_type, errors in predictions.items():
        all_type_errors.append((real_type, pred_type, len(errors), errors))

all_type_errors.sort(key=lambda x: x[2], reverse=True)

for real_type, pred_type, count, errors in all_type_errors[:5]:
    print(f"\n{'─'*100}")
    print(f"CONFUZIE: {real_type} → {pred_type} ({count} erori)")
    print(f"{'─'*100}")
    
    error = errors[0]  # prima eroare
    conv = error["conversation"]
    print(f"\nExemplu: {error['conv_id']}")
    print(f"Reasoning: {error['reasoning']}")
    print("Conversație:")
    for turn in conv.get("turns", []):
        print(f"  {turn['role'].upper()}: {turn['text']}")
    print()

print("\n" + "=" * 100)
print("SFATURI PENTRU FEW-SHOT EXAMPLES")
print("=" * 100)
print("\nBazat pe greșelile de mai sus, caută conversații CORECTE care:")
print("1. Sunt similare cu FP-urile de mai sus dar clasificate corect")
print("2. Acoperă tipurile cu cele mai multe FN")
print("3. Clarifică confuziile între tipuri (ex: incomplet vs irelevant)")
print("\nRulează celula următoare pentru a găsi conversații candidate pentru few-shot.")

SUMAR GREȘELI — openai_o3__en__v3

📊 FALSE POSITIVES (n=4)
----------------------------------------------------------------------------------------------------
  nealiniat_context    →   3 FP
  halucinatie          →   1 FP

📊 FALSE NEGATIVES (n=13)
----------------------------------------------------------------------------------------------------
  halucinatie          →   5 FN
  incomplet            →   4 FN
  irelevant            →   3 FN
  nealiniat_context    →   1 FN

📊 TIP GREȘIT (n=3)
----------------------------------------------------------------------------------------------------
  irelevant            → incomplet            ( 1 erori)
  nealiniat_context    → irelevant            ( 1 erori)
  nealiniat_context    → incomplet            ( 1 erori)


TOP 3 FALSE POSITIVES PER TIP (cu conversații)

────────────────────────────────────────────────────────────────────────────────────────────────────
FP: nealiniat_context (total: 3)
─────────────────────────────────────────────

In [ ]:
# ════════════════════════════════════════════════════════════
# CĂUTARE CONVERSAȚII CANDIDATE PENTRU FEW-SHOT EXAMPLES
# Găsește conversații CORECTE similare cu greșelile frecvente
# ════════════════════════════════════════════════════════════
import json
from pathlib import Path
from collections import defaultdict

# Încarcă experiment și dataset
BEST_EXP = "exp_inc_openai_o3__en__v3.json"
OUTPUT_DIR = Path(r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs_incongruities")
DATASET_PATH = Path(r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\data\master_dataset_refined_180.json")

with open(DATASET_PATH, encoding='utf-8') as f:
    data = json.load(f)
conv_dict = {c["conversation_id"]: c for c in data["conversations"]}

with open(OUTPUT_DIR / BEST_EXP, encoding='utf-8') as f:
    exp_data = json.load(f)

# Găsește conversații CORECTE per tip
correct_by_type = defaultdict(list)

for pred in exp_data["predictions"]:
    if pred.get("parse_failed", False):
        continue
    
    conv_id = pred["conversation_id"]
    ds_has = pred["dataset_has_incongruity"]
    pr_has = pred["predicted_has_incongruity"]
    ds_type = pred.get("dataset_incongruity_type") or "none"
    pr_type = pred.get("predicted_incongruity_type") or "none"
    
    # Clasificare corectă
    if (ds_has == pr_has) and (not ds_has or ds_type == pr_type):
        conv = conv_dict.get(conv_id)
        if conv:
            # Calculează lungimea conversației (nr de turnuri)
            num_turns = len(conv.get("turns", []))
            
            correct_by_type[ds_type].append({
                "conv_id": conv_id,
                "type": ds_type,
                "num_turns": num_turns,
                "conversation": conv,
                "reasoning": pred.get("reasoning", "")
            })

# Sortează conversațiile corecte după lungime (preferăm cele scurte pentru few-shot)
for tip in correct_by_type:
    correct_by_type[tip].sort(key=lambda x: x["num_turns"])

# ══════════════════════════════════════════════════════════════
# AFIȘARE CANDIDATE PER TIP
# ══════════════════════════════════════════════════════════════

# Tipurile în ordinea priorității (bazat pe FN)
priority_types = ["halucinatie", "incomplet", "irelevant", "nealiniat_context", "contradictoriu", "none"]

print("=" * 100)
print("CONVERSAȚII CANDIDATE PENTRU FEW-SHOT EXAMPLES")
print("=" * 100)
print("\nCriteriu: conversații CORECTE clasificate, scurte (2-6 turnuri), clare\n")

candidates = {}

for tip in priority_types:
    if tip not in correct_by_type:
        continue
    
    convs = correct_by_type[tip]
    
    # Filtrează: max 6 turnuri pentru few-shot
    short_convs = [c for c in convs if c["num_turns"] <= 6]
    
    print("=" * 100)
    print(f"TIP: {tip.upper()} (total corecte: {len(convs)}, scurte ≤6 turnuri: {len(short_convs)})")
    print("=" * 100)
    
    if not short_convs:
        print(f"  ⚠️  Nu există conversații scurte pentru {tip}, folosim cele mai scurte disponibile:\n")
        short_convs = convs[:3]
    
    # Afișează primele 3 candidate
    for i, conv_data in enumerate(short_convs[:3], 1):
        conv = conv_data["conversation"]
        print(f"\n[Candidat {i}] {conv_data['conv_id']} ({conv_data['num_turns']} turnuri)")
        print(f"Reasoning: {conv_data['reasoning']}")
        print("Conversație:")
        
        for turn in conv.get("turns", []):
            print(f"  {turn['role'].upper()}: {turn['text']}")
        
        if i == 1:
            # Salvează primul candi

In [27]:
# ════════════════════════════════════════════════════════════
# CĂUTARE CONVERSAȚII CANDIDATE PENTRU FEW-SHOT EXAMPLES
# Găsește conversații CORECTE similare cu greșelile frecvente
# ════════════════════════════════════════════════════════════
import json
from pathlib import Path
from collections import defaultdict

# Încarcă experiment și dataset
BEST_EXP = "exp_inc_openai_o3__en__v3.json"
OUTPUT_DIR = Path(r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs_incongruities")
DATASET_PATH = Path(r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\data\master_dataset_refined_180.json")

with open(DATASET_PATH, encoding='utf-8') as f:
    data = json.load(f)
conv_dict = {c["conversation_id"]: c for c in data["conversations"]}

with open(OUTPUT_DIR / BEST_EXP, encoding='utf-8') as f:
    exp_data = json.load(f)

# Găsește conversații CORECTE per tip
correct_by_type = defaultdict(list)

for pred in exp_data["predictions"]:
    if pred.get("parse_failed", False):
        continue
    
    conv_id = pred["conversation_id"]
    ds_has = pred["dataset_has_incongruity"]
    pr_has = pred["predicted_has_incongruity"]
    ds_type = pred.get("dataset_incongruity_type") or "none"
    pr_type = pred.get("predicted_incongruity_type") or "none"
    
    # Clasificare corectă
    if (ds_has == pr_has) and (not ds_has or ds_type == pr_type):
        conv = conv_dict.get(conv_id)
        if conv:
            # Calculează lungimea conversației (nr de turnuri)
            num_turns = len(conv.get("turns", []))
            
            correct_by_type[ds_type].append({
                "conv_id": conv_id,
                "type": ds_type,
                "num_turns": num_turns,
                "conversation": conv,
                "reasoning": pred.get("reasoning", "")
            })

# Sortează conversațiile corecte după lungime (preferăm cele scurte pentru few-shot)
for tip in correct_by_type:
    correct_by_type[tip].sort(key=lambda x: x["num_turns"])

# ══════════════════════════════════════════════════════════════
# AFIȘARE CANDIDATE PER TIP
# ══════════════════════════════════════════════════════════════

# Tipurile în ordinea priorității (bazat pe FN)
priority_types = ["halucinatie", "incomplet", "irelevant", "nealiniat_context", "contradictoriu", "none"]

print("=" * 100)
print("CONVERSAȚII CANDIDATE PENTRU FEW-SHOT EXAMPLES")
print("=" * 100)
print("\nCriteriu: conversații CORECTE clasificate, scurte (2-6 turnuri), clare\n")

candidates = {}

for tip in priority_types:
    if tip not in correct_by_type:
        continue
    
    convs = correct_by_type[tip]
    
    # Filtrează: max 6 turnuri pentru few-shot
    short_convs = [c for c in convs if c["num_turns"] <= 6]
    
    print("=" * 100)
    print(f"TIP: {tip.upper()} (total corecte: {len(convs)}, scurte ≤6 turnuri: {len(short_convs)})")
    print("=" * 100)
    
    if not short_convs:
        print(f"  ⚠️  Nu există conversații scurte pentru {tip}, folosim cele mai scurte disponibile:\n")
        short_convs = convs[:3]
    
    # Afișează primele 3 candidate
    for i, conv_data in enumerate(short_convs[:3], 1):
        conv = conv_data["conversation"]
        print(f"\n[Candidat {i}] {conv_data['conv_id']} ({conv_data['num_turns']} turnuri)")
        print(f"Reasoning: {conv_data['reasoning']}")
        print("Conversație:")
        
        for turn in conv.get("turns", []):
            print(f"  {turn['role'].upper()}: {turn['text']}")
        
        if i == 1:
            # Salvează primul candidat ca exemplu recomandat
            candidates[tip] = conv_data
        
        print()

# ══════════════════════════════════════════════════════════════
# RECOMANDARE FINALĂ
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 100)
print("📋 RECOMANDARE PENTRU few_shot_examples_incongruities.json")
print("=" * 100)
print("\nConversații recomandate (cele mai scurte și clare):\n")

for tip in priority_types:
    if tip in candidates:
        c = candidates[tip]
        print(f"  {tip:20s} → {c['conv_id']} ({c['num_turns']} turnuri)")

print("\n" + "─" * 100)
print("Următorii pași:")
print("1. Revizuiește candidații de mai sus")
print("2. Selectează 1-2 exemple per tip (prioritate: halucinatie, incomplet, irelevant)")
print("3. Adaugă 1 exemplu 'none' (conversație corectă fără neconcordanță)")
print("4. Creează few_shot_examples_incongruities.json")
print("5. Modifică en_incongruities_v3.jinja → en_incongruities_v4.jinja")
print("=" * 100)

CONVERSAȚII CANDIDATE PENTRU FEW-SHOT EXAMPLES

Criteriu: conversații CORECTE clasificate, scurte (2-6 turnuri), clare

TIP: HALUCINATIE (total corecte: 5, scurte ≤6 turnuri: 4)

[Candidat 1] conv_complex_0077 (4 turnuri)
Reasoning: Asistentul introduce o taxă fixă de 500 lei și un blocaj de 90 zile la închiderea contului în martie, reguli specifice nejustificate în context, sugerând informație inventată.
Conversație:
  USER: Bună, vreau să închid contul meu de economii. Am decis să mă mut în străinătate și nu mai am nevoie de el.
  ASSISTANT: Bună! Înțeleg că doriți să închideți contul de economii. Înainte de a continua, trebuie să știți că, conform regulamentului nostru, orice cont închis în luna martie atrage o taxă de penalizare de 500 de lei, indiferent de sold. De asemenea, fondurile vor fi transferate pe un cont intermediar timp de 90 de zile înainte de a fi restituite. Aveți întrebări despre acest proces?
  USER: Stai, ce taxă de 500 de lei? Nu am auzit de asta înainte. Și 90 d